# EDA LMS Цифриум — структурированная версия

Новый ноутбук повторяет рабочие шаги из `EDA.ipynb` и добавляет понятную структуру для дальнейших экспериментов.

**Структура анализа**

1. Этап подготовки — чтение данных, аудит схем и очистка признаков.
2. Этап слияния — агрегации до уровня `user-course` и описание merge.
3. Этап анализа — описательная статистика и гипотезы по оттоку.

## Этап 1. Подготовка и чистка таблиц

### 1.1 Импорты и глобальные настройки

In [6]:

import pandas as pd
import numpy as np
import json
from pathlib import Path
from IPython.display import display
import warnings

warnings.filterwarnings('ignore')


NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name == 'notebooks' and (NOTEBOOK_DIR.parent / 'data').exists():
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

DATA_DIR_CANDIDATES = [PROJECT_ROOT / 'data' / 'raw', PROJECT_ROOT / 'data']
DATA_DIR = next((p for p in DATA_DIR_CANDIDATES if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError('?? ??????? ????? data/raw ??? data ? CSV ???????????? ????? ???????.')

print(f'Папка проекта: {PROJECT_ROOT}')
print(f'Папка содержащая исходники данных CSV: {DATA_DIR}')

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 120)


Папка проекта: C:\Repos\Xakaton
Папка содержащая исходники данных CSV: C:\Repos\Xakaton\data\raw


### 1.2 Файлы и колонки с датами

In [7]:

FILES = {
    'users_courses': 'users_courses.csv',
    'user_answers': 'user_answers.csv',
    'user_activity_histories': 'user_activity_histories.csv',
    'user_trainings': 'user_trainings.csv',
    'wk_users_courses_actions': 'wk_users_courses_actions.csv',
    'lessons': 'lessons.csv',
    'user_lessons': 'user_lessons.csv',
    'wk_media_view_sessions': 'wk_media_view_sessions.csv',
    'user_access_histories': 'user_access_histories.csv',
    'user_award_badges': 'user_award_badges.csv',
    'users': 'users.csv'
}

DATE_COLS = {
    'users_courses': ['access_finished_at', 'wk_officially_started_at', 'wk_course_completed_at', 'created_at', 'updated_at'],
    'user_answers': ['submitted_at'],
    'user_activity_histories': ['created_at'],
    'user_trainings': ['started_at', 'finished_at', 'mark_saved_at'],
    'wk_users_courses_actions': ['created_at', 'updated_at'],
    'wk_media_view_sessions': ['started_at'],
    'user_access_histories': ['access_started_at', 'access_expired_at'],
    'user_award_badges': ['created_at'],
    'users': ['created_at', 'grade_changed_at', 'updated_at', 'd_updated_at', 'remember_created_at', 'current_sign_in_at', 'last_sign_in_at'],
    'lessons': ['wk_attendance_tracking_disabled_at']
}


### 1.3 Загрузка и первичный аудит

In [8]:

def load_all_tables(data_dir: Path, files: dict, date_cols: dict):
    dfs = {}
    overview = []
    for name, filename in files.items():
        path = data_dir / filename
        if not path.exists():
            print(f'!!! Не найден файл: {path}')
            continue
        parse_dates = date_cols.get(name, [])
        df = pd.read_csv(path, index_col=0, parse_dates=parse_dates, low_memory=False)
        dfs[name] = df
        overview.append({
            'table': name,
            'rows': len(df),
            'cols': df.shape[1],
            'memory_mb': round(df.memory_usage(deep=True).sum() / (1024**2), 2),
            'parsed_dates': len(parse_dates)
        })
    overview_df = pd.DataFrame(overview).sort_values('rows', ascending=False).reset_index(drop=True)
    return dfs, overview_df

dfs, tables_overview = load_all_tables(DATA_DIR, FILES, DATE_COLS)
tables_overview


,table,rows,cols,memory_mb,parsed_dates
0,user_answers,15176182,13,6610.98,1
1,wk_users_courses_actions,12909207,7,3203.60,2
2,user_lessons,3070664,11,844.95,0
3,user_activity_histories,3031137,3,433.17,1
4,wk_media_view_sessions,852358,6,182.51,1
5,user_access_histories,667124,4,87.80,2
6,user_trainings,427628,12,141.96,3
7,users_courses,290835,12,103.34,5
8,user_award_badges,252843,3,21.22,1
9,users,95395,21,70.32,7


### 1.4 Полные дубликаты строк

In [9]:

def summarize_duplicates(dfs: dict) -> pd.DataFrame:
    rows = []
    for name, df in dfs.items():
        dup_count = df.duplicated().sum()
        rows.append({
            'table': name,
            'rows': len(df),
            'duplicates': dup_count,
            'duplicates_%': round((dup_count / len(df)) * 100, 2) if len(df) else 0.0
        })
    summary = pd.DataFrame(rows).sort_values('duplicates_%', ascending=False)
    return summary

duplicate_report = summarize_duplicates(dfs)
duplicate_report


,table,rows,duplicates,duplicates_%
5,lessons,3369,1813,53.81
8,user_access_histories,667124,355372,53.27
4,wk_users_courses_actions,12909207,5490050,42.53
1,user_answers,15176182,4951291,32.63
7,wk_media_view_sessions,852358,22183,2.60
2,user_activity_histories,3031137,33312,1.10
0,users_courses,290835,0,0.00
6,user_lessons,3070664,0,0.00
3,user_trainings,427628,0,0.00
9,user_award_badges,252843,0,0.00


### 1.5 Приведение целочисленных колонок и master-ключей

In [10]:

INT_COLS_MAP = {
    'users_courses': ['user_id', 'course_id'],
    'user_answers': ['user_id', 'task_id'],
    'user_activity_histories': ['user_lesson_id'],
    'user_trainings': ['user_id', 'training_id'],
    'wk_users_courses_actions': ['user_id', 'users_course_id', 'lesson_id'],
    'lessons': ['course_id', 'lesson_number', 'wk_task_count'],
    'user_lessons': ['user_id', 'lesson_id', 'group_id', 'users_course_id'],
    'user_award_badges': ['user_id', 'award_badge_id'],
    'wk_media_view_sessions': ['viewer_id', 'segments_total', 'viewed_segments_count'],
    'users': ['id', 'sign_in_count', 'grade_id', 'd_wk_school_id', 'd_wk_municipal_id', 'd_wk_region_id']
}

def convert_int_like_columns(dfs: dict, int_cols_map: dict):
    for table, cols in int_cols_map.items():
        if table not in dfs:
            continue
        for col in cols:
            if col not in dfs[table].columns:
                continue
            if pd.api.types.is_integer_dtype(dfs[table][col]):
                continue
            dfs[table][col] = pd.to_numeric(
                dfs[table][col].astype(str).str.replace(',', '', regex=False),
                errors='coerce'
            ).astype('Int64')

convert_int_like_columns(dfs, INT_COLS_MAP)

if 'users_courses' in dfs:
    dfs['users_courses'] = dfs['users_courses'].rename_axis('users_course_id').reset_index()


### 1.6 Точечная очистка таблиц

In [11]:

def preprocess_users_courses(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['is_active'] = df['state'].str.lower().eq('active')
    for col in ['wk_max_points', 'wk_max_task_count']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['progress_pct'] = np.where(
        (df['wk_max_points'] > 0) & df['wk_max_points'].notna(),
        (df['wk_points'] / df['wk_max_points'] * 100).round(2),
        np.nan
    )
    df = df.drop(columns=['state', 'wk_points'])
    return df

def preprocess_user_answers(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    bool_cols = ['solved', 'skipped', 'wk_partial_answer']
    for col in bool_cols:
        df[col] = df[col].astype(str).str.strip().str.lower().eq('true')
    df['attempts_pct'] = np.where(
        (df['max_attempts'] > 0) & df['max_attempts'].notna(),
        (df['attempts'] / df['max_attempts'] * 100).round(2),
        np.nan
    )
    df['is_lesson'] = df['resource_type'].eq('Lesson')
    df['is_training'] = df['resource_type'].eq('Training')
    df['is_hw'] = df['resource_type'].eq('Homework')

    def parse_results(value):
        if isinstance(value, str):
            try:
                return json.loads(value)
            except json.JSONDecodeError:
                return []
        return []

    df['parsed_results'] = df['results'].apply(parse_results)
    df = df.explode('parsed_results').reset_index(drop=True)

    def extract_dict(d):
        if not isinstance(d, dict) or not d:
            return (None, None, None)
        key = next(iter(d.keys()))
        payload = next(iter(d.values()))
        return (
            pd.to_numeric(key, errors='coerce'),
            payload.get('points'),
            payload.get('coefficient')
        )

    extracted = df['parsed_results'].apply(extract_dict)
    df['result_subtask_id'] = extracted.apply(lambda x: x[0]).astype('Int64')
    df['result_points'] = pd.to_numeric(extracted.apply(lambda x: x[1]), errors='coerce')
    df['result_coefficient'] = pd.to_numeric(extracted.apply(lambda x: x[2]), errors='coerce')

    df = df.drop(columns=['parsed_results', 'results', 'performance', 'attempts', 'resource_type'])
    return df

def preprocess_user_activity_histories(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['is_visit_video'] = df['action'].eq('visit_video')
    df['is_visit_translation'] = df['action'].eq('visit_translation')
    df['is_show_conspect'] = df['action'].eq('show_conspect')
    df = df.drop(columns=['action'])
    return df

def preprocess_user_trainings(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['is_lesson_training'] = df['type'].eq('UserTrainings::LessonTraining')
    df['is_regular_training'] = df['type'].eq('UserTrainings::RegularTraining')
    df['is_olympiad_training'] = df['type'].eq('UserTrainings::OlympiadTraining')
    df['is_started'] = df['state'].eq('started')
    df = df.drop(columns=['type', 'state'])
    return df

def preprocess_wk_users_courses_actions(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['is_visit_video'] = df['action'].eq('visit_video')
    df['is_show_conspect'] = df['action'].eq('visit_preparation_material')
    df['is_start_training'] = df['action'].eq('start_training')
    df['is_user_answer'] = df['action'].eq('user_answer')
    df['is_visit_translation'] = df['action'].eq('visit_translation')
    df['is_scratch_playground_visited'] = df['action'].eq('scratch_playground_visited')
    df = df.drop(columns=['action', 'sourceable_id', 'lesson_id'])
    return df

def preprocess_user_award_badges(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    pivot = df.pivot_table(index='user_id', columns='award_badge_id', values='created_at', aggfunc='min')
    date_cols = {f'date_badge_{int(c)}': pivot[c] for c in pivot.columns}
    dates = pd.DataFrame(date_cols)
    flag_cols = {f'has_badge_{int(c)}': pivot[c].notna() for c in pivot.columns}
    flags = pd.DataFrame(flag_cols)
    features = pd.concat([flags, dates], axis=1).reset_index()
    return features

def preprocess_users(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['user_id'] = df['id']
    drop_cols = ['is_teacher', 'current_sign_in_at', 'last_sign_in_at', 'grade_checked', 'xp', 'd_updated_at']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    df['is_teacher'] = df['type'].str.contains('Agent', na=False)
    df = df.drop(columns=['type'])
    return df

def preprocess_user_access_histories(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['is_premium_access'] = df['activator_class'].eq('Courses::AccessActivators::PremiumAccessActivator')
    df['is_revoke_access'] = df['activator_class'].eq('Courses::AccessActivators::RevokeAccessActivator')
    df['is_standard_access'] = df['activator_class'].eq('Courses::AccessActivators::StandardAccessActivator')
    df['is_change_duration_access'] = df['activator_class'].eq('Courses::AccessActivators::ChangeAccessDurationActivator')
    df['is_month_premium_access'] = df['activator_class'].eq('Courses::AccessActivators::MonthPremiumAccessActivator')
    df = df.drop(columns=['activator_class'])
    return df

def preprocess_user_lessons(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    bool_cols = ['video_visited', 'translation_visited', 'solved', 'video_viewed']
    for col in bool_cols:
        if col in df.columns:
            df[col] = df[col].fillna(False).astype(bool)
    return df

def preprocess_lessons(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    bool_cols = ['conspect_expected', 'task_expected', 'wk_survival_training_expected',
                 'wk_scratch_playground_enabled', 'wk_attendance_tracking_enabled']
    for col in bool_cols:
        if col in df.columns:
            df[col] = df[col].fillna(False).astype(bool)
    df['wk_video_duration'] = pd.to_numeric(df['wk_video_duration'], errors='coerce')
    return df

def preprocess_wk_media_view_sessions(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['segments_total'] = pd.to_numeric(df['segments_total'], errors='coerce')
    df['viewed_segments_count'] = pd.to_numeric(df['viewed_segments_count'], errors='coerce')
    return df

TABLE_PREPROCESSORS = {
    'users_courses': preprocess_users_courses,
    'user_answers': preprocess_user_answers,
    'user_activity_histories': preprocess_user_activity_histories,
    'user_trainings': preprocess_user_trainings,
    'wk_users_courses_actions': preprocess_wk_users_courses_actions,
    'user_award_badges': preprocess_user_award_badges,
    'users': preprocess_users,
    'user_access_histories': preprocess_user_access_histories,
    'user_lessons': preprocess_user_lessons,
    'lessons': preprocess_lessons,
    'wk_media_view_sessions': preprocess_wk_media_view_sessions
}

for name, func in TABLE_PREPROCESSORS.items():
    if name in dfs:
        dfs[name] = func(dfs[name])


### 1.7 Быстрый дата-качественный отчёт по ключевым таблицам

In [13]:

def eda_report(dfs_subset: dict, max_cat_examples: int = 3):
    with pd.option_context('display.max_columns', None, 'display.width', 120):
        for name, df in dfs_subset.items():
            print(f"\n{'='*20} {name} {'='*20}")
            mem_mb = df.memory_usage(deep=True).sum() / (1024**2)
            missing_total = df.isnull().sum().sum()
            missing_pct = (missing_total / (df.shape[0] * max(df.shape[1], 1))) * 100
            print(f"Shape: {df.shape[0]:,} x {df.shape[1]} | Память: {mem_mb:.2f} MB | Пропуски: {missing_total:,} ({missing_pct:.2f}%)")

            num_cols = df.select_dtypes(include=['number']).columns
            if len(num_cols) > 0:
                num_stats = df[num_cols].describe(percentiles=[0.5]).T[['count', 'mean', 'std', 'min', '50%', 'max']]
                num_stats['missing_%'] = (df[num_cols].isnull().mean() * 100).round(0)
                print('Числовые признаки:')
                print(num_stats.to_string())

            bool_cols = df.select_dtypes(include='bool').columns
            if len(bool_cols) > 0:
                rows = []
                for col in bool_cols:
                    vc = df[col].value_counts(normalize=True)
                    rows.append({
                        'feature': col,
                        'true_%': round(vc.get(True, 0) * 100, 1),
                        'false_%': round(vc.get(False, 0) * 100, 1)
                    })
                print('Булевые признаки:')
                print(pd.DataFrame(rows).to_string(index=False))

            cat_cols = df.select_dtypes(include=['object']).columns
            if len(cat_cols) > 0:
                rows = []
                for col in cat_cols:
                    vc = df[col].value_counts(normalize=True).head(max_cat_examples)
                    rows.append({
                        'feature': col,
                        'unique': df[col].nunique(dropna=True),
                        'top_values': ' | '.join([f"{idx} ({val:.0%})" for idx, val in vc.items()])
                    })
                print('Категориальные признаки:')
                print(pd.DataFrame(rows).to_string(index=False))

key_tables = ['users_courses', 'user_answers', 'wk_users_courses_actions', 'user_lessons']
eda_report({name: dfs[name] for name in key_tables if name in dfs})



==================== users_courses ====================
Shape: 290,835 x 13 | Память: 27.46 MB | Пропуски: 571,367 (15.11%)
Числовые признаки:
                            count            mean              std       min       50%          max  missing_%
users_course_id          290835.0        145417.0     83956.977107       0.0  145417.0     290834.0        0.0
user_id                  290835.0   715722.751409     27293.035063  665740.0  716070.0     761578.0        0.0
course_id                290835.0  6916813.866577  29739455.750645     754.0     771.0  170000688.0        0.0
wk_max_points            290699.0       78.535638        89.266805       0.0      63.0        617.0        0.0
wk_max_viewable_lessons  290710.0       12.288298         7.222627       0.0      14.0         80.0        0.0
wk_max_task_count        290699.0       81.990764        90.913452       0.0      63.0        524.0        0.0
progress_pct             204410.0       70.158774        26.570576       0.0   

## Этап 2. Слияние таблиц и подготовка фичей

На этом этапе все источники приводим к уровню `user-course` и фиксируем предположения по ключам.

In [ ]:

def aggregate_lessons(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=['course_id'])
    agg = df.groupby('course_id').agg(
        lessons_total=('lesson_number', 'nunique'),
        lessons_with_conspect_share=('conspect_expected', 'mean'),
        lessons_with_tasks_share=('task_expected', 'mean'),
        lessons_with_survival_share=('wk_survival_training_expected', 'mean'),
        lessons_with_scratch_share=('wk_scratch_playground_enabled', 'mean'),
        total_lesson_points=('wk_max_points', 'sum'),
        total_lesson_tasks=('wk_task_count', 'sum'),
        total_video_minutes=('wk_video_duration', 'sum')
    ).reset_index()
    pct_cols = [c for c in agg.columns if c.endswith('_share')]
    for col in pct_cols:
        agg[col] = (agg[col] * 100).round(1)
    return agg

def aggregate_user_lessons(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=['users_course_id'])
    agg = df.groupby('users_course_id').agg(
        lessons_touched=('lesson_id', 'nunique'),
        lessons_solved=('solved', 'sum'),
        solved_tasks_total=('wk_solved_task_count', 'sum'),
        lesson_points_total=('wk_points', 'sum'),
        video_visited_share=('video_visited', 'mean'),
        translation_visited_share=('translation_visited', 'mean')
    ).reset_index()
    agg['lessons_solved_share'] = np.where(
        agg['lessons_touched'] > 0,
        (agg['lessons_solved'] / agg['lessons_touched'] * 100).round(1),
        np.nan
    )
    agg['video_visited_share'] = (agg['video_visited_share'] * 100).round(1)
    agg['translation_visited_share'] = (agg['translation_visited_share'] * 100).round(1)
    return agg

def aggregate_course_actions(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=['users_course_id'])
    agg = df.groupby('users_course_id').agg(
        total_actions=('users_course_id', 'size'),
        first_action_at=('created_at', 'min'),
        last_action_at=('created_at', 'max'),
        visit_video_count=('is_visit_video', 'sum'),
        show_conspect_count=('is_show_conspect', 'sum'),
        start_training_count=('is_start_training', 'sum'),
        user_answer_count=('is_user_answer', 'sum'),
        visit_translation_count=('is_visit_translation', 'sum'),
        scratch_playground_count=('is_scratch_playground_visited', 'sum')
    ).reset_index()
    agg['active_days'] = (agg['last_action_at'] - agg['first_action_at']).dt.days
    return agg

def aggregate_user_answers(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=['user_id'])
    agg = df.groupby('user_id').agg(
        answers_total=('task_id', 'count'),
        solved_share=('solved', 'mean'),
        skipped_share=('skipped', 'mean'),
        lesson_answer_share=('is_lesson', 'mean'),
        training_answer_share=('is_training', 'mean'),
        hw_answer_share=('is_hw', 'mean'),
        avg_attempts_pct=('attempts_pct', 'mean'),
        avg_result_points=('result_points', 'mean')
    ).reset_index()
    share_cols = ['solved_share', 'skipped_share', 'lesson_answer_share', 'training_answer_share', 'hw_answer_share']
    for col in share_cols:
        agg[col] = (agg[col] * 100).round(1)
    return agg

def aggregate_user_access_histories(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=['users_course_id'])
    agg = df.groupby('users_course_id').agg(
        access_events=('users_course_id', 'size'),
        first_access_at=('access_started_at', 'min'),
        last_access_at=('access_expired_at', 'max'),
        premium_access_count=('is_premium_access', 'sum'),
        revoke_access_count=('is_revoke_access', 'sum'),
        standard_access_count=('is_standard_access', 'sum'),
        month_premium_count=('is_month_premium_access', 'sum'),
        change_duration_count=('is_change_duration_access', 'sum')
    ).reset_index()
    agg['access_duration_days'] = (agg['last_access_at'] - agg['first_access_at']).dt.days
    return agg

def aggregate_user_trainings(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=['user_id'])
    agg = df.groupby('user_id').agg(
        trainings_total=('training_id', 'nunique'),
        trainings_started_share=('is_started', 'mean'),
        lesson_trainings=('is_lesson_training', 'sum'),
        regular_trainings=('is_regular_training', 'sum'),
        olympiad_trainings=('is_olympiad_training', 'sum'),
        total_training_points=('earned_points', 'sum'),
        total_training_attempts=('attempts', 'sum')
    ).reset_index()
    agg['trainings_started_share'] = (agg['trainings_started_share'] * 100).round(1)
    return agg

def enrich_users_with_badges(users_df: pd.DataFrame, badges_df: pd.DataFrame) -> pd.DataFrame:
    users_df = users_df.copy()
    if badges_df is None or badges_df.empty:
        return users_df
    merged = users_df.merge(badges_df, on='user_id', how='left')
    badge_cols = [col for col in merged.columns if col.startswith('has_badge_')]
    for col in badge_cols:
        merged[col] = merged[col].fillna(False)
    return merged


In [ ]:

lessons_course_features = aggregate_lessons(dfs.get('lessons', pd.DataFrame()))
user_lessons_features = aggregate_user_lessons(dfs.get('user_lessons', pd.DataFrame()))
course_actions_features = aggregate_course_actions(dfs.get('wk_users_courses_actions', pd.DataFrame()))
user_answers_features = aggregate_user_answers(dfs.get('user_answers', pd.DataFrame()))
user_access_features = aggregate_user_access_histories(dfs.get('user_access_histories', pd.DataFrame()))
user_training_features = aggregate_user_trainings(dfs.get('user_trainings', pd.DataFrame()))
users_enriched = enrich_users_with_badges(dfs.get('users', pd.DataFrame()), dfs.get('user_award_badges', pd.DataFrame()))

feature_shapes = pd.DataFrame({
    'dataset': [
        'lessons_course_features', 'user_lessons_features', 'course_actions_features',
        'user_answers_features', 'user_access_features', 'user_training_features', 'users_enriched'
    ],
    'shape': [
        lessons_course_features.shape,
        user_lessons_features.shape,
        course_actions_features.shape,
        user_answers_features.shape,
        user_access_features.shape,
        user_training_features.shape,
        users_enriched.shape
    ]
})
feature_shapes


### Merge-шаги и договорённости

In [ ]:

user_course_features = dfs['users_courses'].copy()

merge_steps = [
    {'right': lessons_course_features, 'on': 'course_id', 'validate': 'many_to_one', 'note': 'course metadata (lessons)'},
    {'right': user_lessons_features, 'on': 'users_course_id', 'validate': 'one_to_one', 'note': 'user lesson progress'},
    {'right': course_actions_features, 'on': 'users_course_id', 'validate': 'one_to_one', 'note': 'wk actions log'},
    {'right': user_access_features, 'on': 'users_course_id', 'validate': 'one_to_one', 'note': 'access history'},
    {'right': users_enriched, 'on': 'user_id', 'validate': 'many_to_one', 'note': 'user profile + badges'},
    {'right': user_answers_features, 'on': 'user_id', 'validate': 'many_to_one', 'note': 'task-level answers (assumption user_id <-> course)'},
    {'right': user_training_features, 'on': 'user_id', 'validate': 'many_to_one', 'note': 'trainings activity (assumption user_id <-> course)'}
]

merge_log = []
for step in merge_steps:
    right = step['right']
    if right is None or right.empty:
        merge_log.append({'note': step['note'], 'status': 'пропущено', 'reason': 'пустой датасет'})
        continue
    before_cols = user_course_features.shape[1]
    user_course_features = user_course_features.merge(
        right,
        on=step['on'],
        how='left',
        validate=step.get('validate', 'many_to_one')
    )
    merge_log.append({
        'note': step['note'],
        'status': 'успех',
        'new_columns': user_course_features.shape[1] - before_cols
    })

pd.DataFrame(merge_log)


In [ ]:

user_course_features['lessons_solved_share'] = np.where(
    user_course_features['lessons_touched'] > 0,
    (user_course_features['lessons_solved'] / user_course_features['lessons_touched'] * 100).round(1),
    np.nan
)
user_course_features['inactivity_gap_days'] = (
    user_course_features['access_finished_at'] - user_course_features['last_action_at']
).dt.days
user_course_features['preexpiry_inactivity_flag'] = user_course_features['inactivity_gap_days'].gt(14)
user_course_features['access_to_start_days'] = (
    user_course_features['access_finished_at'] - user_course_features['wk_officially_started_at']
).dt.days

user_course_features.head()


## Этап 3. Анализ подготовленных таблиц

### 3.1 Обзор итогового датасета

In [ ]:

final_overview = {
    'rows': user_course_features.shape[0],
    'columns': user_course_features.shape[1],
    'memory_mb': round(user_course_features.memory_usage(deep=True).sum() / (1024**2), 2)
}
print(final_overview)
user_course_features.info(verbose=False)


### 3.2 Пропуски

In [ ]:

missing_pct = (user_course_features.isnull().mean() * 100).sort_values(ascending=False)
missing_pct.head(25)


### 3.3 Распределения ключевых метрик

In [ ]:

key_metrics = ['progress_pct', 'lessons_touched', 'lessons_solved_share', 'total_actions', 'active_days', 'access_duration_days', 'inactivity_gap_days']
available_metrics = [col for col in key_metrics if col in user_course_features.columns]
if available_metrics:
    metrics_summary = user_course_features[available_metrics].describe(percentiles=[0.25, 0.5, 0.75, 0.9]).T
else:
    metrics_summary = pd.DataFrame(index=available_metrics)
metrics_summary


### 3.4 Активные и неактивные

In [ ]:

state_cols = ['progress_pct', 'total_actions', 'lessons_solved_share', 'inactivity_gap_days']
available_state_cols = [col for col in state_cols if col in user_course_features.columns]
if available_state_cols:
    state_summary = user_course_features.groupby('is_active')[available_state_cols].median()
else:
    state_summary = pd.DataFrame(columns=available_state_cols)
state_summary


### 3.5 Spot-check user-course 665744 / 50000592

In [ ]:

EXAMPLE_USER_ID = 665744
EXAMPLE_COURSE_ID = 50000592
EXAMPLE_USERS_COURSE_ID = 449093

lessons_df = dfs.get('lessons', pd.DataFrame())
if not lessons_df.empty and 'course_id' in lessons_df.columns:
    lessons_sample = lessons_df[lessons_df['course_id'] == EXAMPLE_COURSE_ID]
else:
    lessons_sample = lessons_df.head(0)

wk_actions_df = dfs.get('wk_users_courses_actions', pd.DataFrame())
if not wk_actions_df.empty and 'users_course_id' in wk_actions_df.columns:
    wk_actions_sample = wk_actions_df[wk_actions_df['users_course_id'] == EXAMPLE_USERS_COURSE_ID]
else:
    wk_actions_sample = wk_actions_df.head(0)

user_answers_df = dfs.get('user_answers', pd.DataFrame())
if not user_answers_df.empty and 'user_id' in user_answers_df.columns:
    user_answers_sample = user_answers_df[user_answers_df['user_id'] == EXAMPLE_USER_ID]
else:
    user_answers_sample = user_answers_df.head(0)

user_trainings_df = dfs.get('user_trainings', pd.DataFrame())
if not user_trainings_df.empty and 'user_id' in user_trainings_df.columns:
    user_trainings_sample = user_trainings_df[user_trainings_df['user_id'] == EXAMPLE_USER_ID]
else:
    user_trainings_sample = user_trainings_df.head(0)

user_access_df = dfs.get('user_access_histories', pd.DataFrame())
if not user_access_df.empty and 'users_course_id' in user_access_df.columns:
    user_access_sample = user_access_df[user_access_df['users_course_id'] == EXAMPLE_USERS_COURSE_ID]
else:
    user_access_sample = user_access_df.head(0)

examples = {
    'users_courses_row': user_course_features.query('users_course_id == @EXAMPLE_USERS_COURSE_ID'),
    'lessons_for_course': lessons_sample,
    'wk_actions': wk_actions_sample,
    'user_answers': user_answers_sample,
    'user_trainings': user_trainings_sample,
    'user_access_histories': user_access_sample
}
for name, frame in examples.items():
    print(f"
{name} -> shape {frame.shape}")
    display(frame.head())


### 3.6 Гипотезы по оттоку

1. **Долгий перерыв перед окончанием доступа.** `preexpiry_inactivity_flag` + низкий `progress_pct` сигнализируют о высоком риске.
2. **Активность без прогресса.** Большой `total_actions` и низкий `lessons_solved_share` показывают блуждение по курсу.
3. **Курсы без методической поддержки.** Низкие `lessons_with_conspect_share` + высокая доля заданий усиливают отток.
4. **Мало наград и дней активности.** `has_badge_* == False` в сочетании с коротким `active_days` даёт сильный сигнал о скором оттоке.
5. **Сегмент учителей.** Флаг `is_teacher` позволяет снимать нетипичных пользователей из анализа.